# Day and Timestamp Feature Analysis

This notebook analyzes the `day` and `timestamp` features in the dataset. Specifically, it:
1. Counts the number of unique days and timestamp values in both `train.csv` and `test.csv`.
2. Evaluates the individual and joint information content of these features with respect to the target variable `demand`.
   - **Predictive Power**: Measured using 5-Fold Cross-Validation R2 score of a Gradient Boosting model.
   - **Information Theory**: Measured using Mutual Information (MI) regression score.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# Load train and test data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# Convert timestamp to minute of day (tmin)
train['tmin'] = train['timestamp'].map(lambda s: int(s.split(':')[0]) * 60 + int(s.split(':')[1]))
test['tmin'] = test['timestamp'].map(lambda s: int(s.split(':')[0]) * 60 + int(s.split(':')[1]))

## 1. Unique Days and Timestamp Values

In [ ]:
print("--- Unique Days & Timestamps ---")
print(f"Train - Unique days: {train['day'].nunique()}, values: {train['day'].unique().tolist()}")
print(f"Train - Unique timestamp values: {train['timestamp'].nunique()}")
print(f"Test - Unique days: {test['day'].nunique()}, values: {test['day'].unique().tolist()}")
print(f"Test - Unique timestamp values: {test['timestamp'].nunique()}")

## 2. Predictive Power (5-Fold CV R2 Score)
We evaluate how well `day`, `timestamp` (as `tmin`), and their combination can predict `demand` using `HistGradientBoostingRegressor`.

In [ ]:
y = train['demand'].values

def evaluate_features(feature_cols):
    kf = KFold(5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(train))
    X = train[feature_cols].copy()
    
    # If using string columns, one-hot encode
    if any(train[col].dtype == object for col in feature_cols):
        X = pd.get_dummies(X, columns=[col for col in feature_cols if train[col].dtype == object], drop_first=True)
        
    X_val = X.values
    
    for tr_idx, val_idx in kf.split(X):
        model = HistGradientBoostingRegressor(random_state=42, max_iter=100)
        model.fit(X_val[tr_idx], y[tr_idx])
        oof_preds[val_idx] = model.predict(X_val[val_idx])
        
    return r2_score(y, oof_preds)

r2_day = evaluate_features(['day'])
r2_tmin = evaluate_features(['tmin'])
r2_joint = evaluate_features(['day', 'tmin'])

print(f"R2 (day only): {r2_day:.6f}")
print(f"R2 (timestamp/tmin only): {r2_tmin:.6f}")
print(f"R2 (day + timestamp/tmin jointly): {r2_joint:.6f}")

## 3. Mutual Information (MI)
Mutual Information measures the amount of information that can be obtained about the target variable `demand` through the features.

In [ ]:
# Calculate MI
mi_day = mutual_info_regression(train[['day']], y, random_state=42)[0]
mi_tmin = mutual_info_regression(train[['tmin']], y, random_state=42)[0]

# Joint MI: Concatenate day and tmin as a joint categorical variable code
train['joint_day_tmin'] = train['day'].astype(str) + "_" + train['tmin'].astype(str)
train['joint_day_tmin_code'] = train['joint_day_tmin'].astype('category').cat.codes
mi_joint = mutual_info_regression(train[['joint_day_tmin_code']], y, random_state=42)[0]

print(f"Mutual Info - day: {mi_day:.6f}")
print(f"Mutual Info - timestamp/tmin: {mi_tmin:.6f}")
print(f"Mutual Info - (day, timestamp) Jointly: {mi_joint:.6f}")

## Summary & Interpretation

1. **Unique Values**:
   - `train.csv` contains two days: **Day 48** (full day) and **Day 49** (morning only, up to 02:00 / minute 120).
   - `test.csv` contains only one day: **Day 49** (daytime only, 02:15 to 13:45 / minutes 135 to 825).
   - `train` covers 96 distinct timestamp values (every 15 mins for 24h), while `test` covers 47 distinct timestamp values (daytime window).

2. **Feature Significance & Joint Info**:
   - **`day` alone** has near-zero predictive power (R2 = 0.000651, MI = 0.000000). This is because the two days cover completely different time windows.
   - **`timestamp` (tmin) alone** is a significant feature (R2 = 0.015473, MI = 0.021668). The demand strongly depends on the time of day.
   - **Jointly (`day` + `timestamp`)**, there is a significant boost (R2 increases to 0.019035, MI increases to 0.024128). This indicates that the demand patterns across timestamps are not identical between Day 48 and Day 49, and knowing the combination of day and time provides more information than either feature individually.

## 4. Time-Series Analysis per Geohash

To determine if this is a time-series dataset and how data points are distributed, we analyze the record density per geohash and visualize their temporal activity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure train is loaded
if 'train' not in globals():
    train = pd.read_csv('train.csv')
    train['tmin'] = train['timestamp'].map(lambda s: int(s.split(':')[0]) * 60 + int(s.split(':')[1]))

# Combine day and tmin to create a continuous time index
train['time_idx'] = (train['day'] - 48) * 1440 + train['tmin']

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Histogram of points per geohash
points_per_geohash = train.groupby('geohash').size()
axes[0, 0].hist(points_per_geohash, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution of Data Points per Geohash (Max=105)', fontsize=14)
axes[0, 0].set_xlabel('Number of Data Points in train.csv')
axes[0, 0].set_ylabel('Count of Geohashes')
axes[0, 0].grid(True, linestyle='--', alpha=0.6)

# 2. Activity Grid for 50 Sampled Geohashes
sample_geohashes = points_per_geohash.sample(50, random_state=42).index
sample_df = train[train['geohash'].isin(sample_geohashes)]
activity_matrix = sample_df.pivot_table(
    index='geohash', columns='time_idx', values='demand', aggfunc='count'
).fillna(0)
activity_matrix = activity_matrix.reindex(sorted(activity_matrix.columns), axis=1)

im = axes[0, 1].imshow(activity_matrix, aspect='auto', cmap='Blues', interpolation='nearest')
axes[0, 1].set_title('Activity Grid for 50 Sampled Geohashes (Blue = Active, White = Missing)', fontsize=14)
axes[0, 1].set_xlabel('Time Index (Minutes from Day 48 start)')
axes[0, 1].set_ylabel('Geohash Index')
axes[0, 1].set_yticks(range(len(activity_matrix.index)))
axes[0, 1].set_yticklabels(activity_matrix.index, fontsize=6)
fig.colorbar(im, ax=axes[0, 1], label='Activity Count')

# 3. Overall temporal density
active_geohashes_per_time = train.groupby(['day', 'tmin']).size().reset_index(name='active_count')
active_geohashes_per_time['time_idx'] = (active_geohashes_per_time['day'] - 48) * 1440 + active_geohashes_per_time['tmin']
active_geohashes_per_time = active_geohashes_per_time.sort_values('time_idx')

axes[1, 0].plot(active_geohashes_per_time['time_idx'], active_geohashes_per_time['active_count'], color='teal', linewidth=2)
axes[1, 0].axvline(x=1440, color='red', linestyle='--', label='Start of Day 49')
axes[1, 0].set_title('Number of Active Geohashes per Timestamp', fontsize=14)
axes[1, 0].set_xlabel('Time Index (Minutes from Day 48 start)')
axes[1, 0].set_ylabel('Active Geohash Count')
axes[1, 0].legend()
axes[1, 0].grid(True, linestyle='--', alpha=0.6)

# 4. Time series demand for top 3 most active geohashes
top_3_geohashes = points_per_geohash.nlargest(3).index
for gh in top_3_geohashes:
    gh_data = train[train['geohash'] == gh].sort_values('time_idx')
    axes[1, 1].plot(gh_data['time_idx'], gh_data['demand'], label=f'Geohash: {gh}', alpha=0.8)
axes[1, 1].axvline(x=1440, color='red', linestyle='--', label='Start of Day 49')
axes[1, 1].set_title('Demand Time-Series for Top 3 Most Active Geohashes', fontsize=14)
axes[1, 1].set_xlabel('Time Index (Minutes from Day 48 start)')
axes[1, 1].set_ylabel('Demand')
axes[1, 1].legend()
axes[1, 1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

### Geohash Time-Series Characteristics

1. **Is it a time-series dataset?**
   - Yes, it is a time-series dataset. Each record is indexed by a specific `geohash`, `day`, and `timestamp`.
   - However, it is **highly sparse and irregularly sampled** for many geohashes. 
   - Only a small fraction of geohashes are active at all 105 available timestamps. The median geohash has only 71 points (out of a maximum 105), and 25% of geohashes have 30 or fewer points. Some geohashes only have a single record in the entire training set.

2. **Data Point Density and Distribution**:
   - **Point Count Distribution**: The histogram shows that while some geohashes have the full 105 points, the majority are scattered, with a significant group having low density.
   - **Activity Grid**: The heatmap shows the sampling sparsity visually. While some rows (geohashes) are solid blue (continuous measurements), many rows are dotted with white space, indicating missing timestamps where no demands were recorded.
   - **Overall Active Geohashes**: The count of active geohashes per timestamp is relatively stable across day 48 but drops slightly during the late night, and then continues into the day 49 morning.

1249

# 5. Feature Engineering and Correlation Analysis

We computed all features for both:
- **Option A**: The standard training subset (`tr_tgt`), which uses leak-free Out-Of-Fold (OOF) analog feature engineering.
- **Option B**: The entire `train.csv`, which uses the full analog feature engineering (built on Day 48).

Then, we computed the Pearson correlation matrix and the absolute value correlation matrix (scaled to 0-1 range) for all features, including the target variable `demand`.

In [ ]:
import pandas as pd
import numpy as np

# Load the absolute correlation matrices
abs_corr_training = pd.read_csv('abs_correlation_matrix_training_set.csv', index_col=0)
abs_corr_whole = pd.read_csv('abs_correlation_matrix_whole_train.csv', index_col=0)

print("--- Top 15 Features Correlated with TARGET (demand) in Training Set (Option A) ---")
print(abs_corr_training['demand'].sort_values(ascending=False).head(16))

print("\n--- Top 15 Features Correlated with TARGET (demand) in Whole train.csv (Option B) ---")
print(abs_corr_whole['demand'].sort_values(ascending=False).head(16))

## Visualizing the Correlation Matrix (0 to 1 scale) for all features

Here we visualize the absolute correlation matrix for the whole `train.csv` (Option B).

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(18, 14))
sns.heatmap(abs_corr_whole, annot=False, cmap='viridis', vmin=0, vmax=1)
plt.title('Absolute Correlation Matrix (0 to 1 Scale) - Whole train.csv', fontsize=16)
plt.tight_layout()
plt.show()

# 6. Test Set Feature Engineering and Correlation Analysis

We computed all features for the test set (`test.csv`) using the history from `train.csv` (mirroring the test inference pipeline in `curr_best91.py`). We then aligned the true target variable `demand` from `real_test.csv` using the unique `Index` column.

Below we load and analyze the absolute correlation matrix for all features on the test set.

In [ ]:
import pandas as pd
import numpy as np

# Load the test set absolute correlation matrix
abs_corr_test = pd.read_csv('abs_correlation_matrix_test_set.csv', index_col=0)

print("--- Top 15 Features Correlated with TARGET (demand) in Test Set ---")
print(abs_corr_test['demand'].sort_values(ascending=False).head(16))

## Visualizing the Test Set Correlation Matrix (0 to 1 scale) for all features

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(18, 14))
sns.heatmap(abs_corr_test, annot=False, cmap='viridis', vmin=0, vmax=1)
plt.title('Absolute Correlation Matrix (0 to 1 Scale) - Test Set (Day 49)', fontsize=16)
plt.tight_layout()
plt.show()